# LABORATORIO DIRIGIDO N.° 03 — SQL avanzado
**Semana 03 — Fundamentos de Gestión de Datos, TECSUP 2026-II.**  
**Docente:** Pilar Rocío Sayán Mejía.  
**Caso:** TecnoAndina Retail.  
**Equipo / sección:** completar con los datos reales del equipo antes de entregar.

**Fuente:** https://github.com/Rociosayan/Repo_FDG_2026-2/tree/main/semana03/datos

*Este notebook se preparó a partir de una copia de la actividad D3 aportada por el estudiante. Se corrigieron y completaron sus respuestas; se conservaron las salidas ya visibles en esa copia, por lo que no deben interpretarse como una nueva ejecución verificada. Ejecútese de principio a fin en Colab para actualizarlas.*

## Caso de estudio: TecnoAndina Retail

TecnoAndina Retail es una cadena peruana de tecnología con tiendas en Lima, Arequipa, Trujillo y Piura y un canal de venta por Web. Su información está repartida en cuatro tablas y el área comercial necesita saber qué clientes compran más, cuáles todavía no compran y qué productos y categorías sustentan las ventas.

Naturaleza de los datos. Conjunto de práctica inventado (no son ventas ni personas reales), con pedidos del 1 de agosto al 30 de septiembre de 2026, publicado en GitHub en LD3_TecnoAndina_Retail.xlsx. Sirve para practicar consultas, no para afirmar tendencias ni rentabilidad.

- clientes: id_cliente, nombre, ciudad. Una fila es un cliente registrado.

- productos: id_producto, nombre_producto, categoria, precio_lista. Una fila es un producto del catálogo.

- pedidos: id_pedido, id_cliente, fecha_pedido, canal, total. Una fila es un pedido de un cliente.

- detalles_pedido: id_detalle, id_pedido, id_producto, cantidad, precio_unitario. Una fila es un producto incluido en un pedido.

Las tablas se relacionan por id_cliente (clientes–pedidos), id_pedido (pedidos–detalles_pedido) e id_producto (productos–detalles_pedido).

### Preparación del entorno

Abra el notebook en Google Colab y ejecute la celda 1: importa sqlite3 y pandas, crea la base en memoria, carga las cuatro hojas del Excel desde GitHub y cuenta los registros. Cada resultado se muestra con pd.read_sql_query; pandas muestra los NULL como NaN o None. Si reinicia el entorno, vuelva a ejecutar la celda 1.

**Celda 1 de 12**

In [ ]:
import sqlite3
import pandas as pd

# Base de práctica de TecnoAndina Retail (agosto y septiembre de 2026)
url = "https://raw.githubusercontent.com/Rociosayan/Repo_FDG_2026-2/main/semana03/datos/LD3_TecnoAndina_Retail.xlsx"
conexion = sqlite3.connect(":memory:")

# Cada hoja del Excel pasa a una tabla de SQLite
pd.read_excel(url, sheet_name="clientes").to_sql("clientes", conexion, index=False)
pd.read_excel(url, sheet_name="productos").to_sql("productos", conexion, index=False)
pd.read_excel(url, sheet_name="pedidos").to_sql("pedidos", conexion, index=False)
pd.read_excel(url, sheet_name="detalles_pedido").to_sql("detalles_pedido", conexion, index=False)

consulta = """SELECT
  (SELECT COUNT(*) FROM clientes) AS clientes,
  (SELECT COUNT(*) FROM productos) AS productos,
  (SELECT COUNT(*) FROM pedidos) AS pedidos,
  (SELECT COUNT(*) FROM detalles_pedido) AS detalles_pedido;"""
pd.read_sql_query(consulta, conexion)

   clientes  productos  pedidos  detalles_pedido
0        24         14       42               68

## A. Relacionar tablas

### Ejercicio 1 — INNER JOIN y LEFT JOIN: clientes con y sin pedidos

El área comercial quiere saber cuántos clientes compraron y quiénes todavía no registran pedidos. INNER JOIN conserva solo las filas con pareja; LEFT JOIN conserva todas las filas de la tabla izquierda y completa con NULL.

Se solicita. Ejecute las celdas 2 y 3.

**Celda 2 de 12**

In [ ]:
consulta = """SELECT COUNT(p.id_pedido) AS pedidos_unidos,
       COUNT(DISTINCT c.id_cliente) AS clientes_con_pedidos,
       (SELECT COUNT(*) FROM clientes) AS clientes_registrados
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

   pedidos_unidos  clientes_con_pedidos  clientes_registrados
0              42                    18                    24

**Celda 3 de 12**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
ORDER BY c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

   id_cliente             nombre    ciudad
0           6  Andrés Villanueva      Lima
1          11     Raúl Benavides  Arequipa
2          15       Héctor Burga  Trujillo
3          20      Marta Infante     Piura
4          22     Gabriela Ponce  Arequipa
5          24        Natalia Ato     Piura

### Respuestas — INNER JOIN y LEFT JOIN

**Pregunta 1.** La celda 2 devuelve **42 pedidos**, correspondientes a **18 clientes distintos**, de **24 registrados**. INNER JOIN exige coincidencias en ambas tablas y excluye a los seis clientes sin pedido; COUNT(DISTINCT id_cliente) evita contar al mismo cliente varias veces.

**Pregunta 2.** La tabla izquierda es **clientes**. LEFT JOIN mantiene sus filas, incluso sin pedidos; WHERE p.id_pedido IS NULL deja únicamente los **6 sin compras**. En SQL se usa IS NULL, no = NULL, porque la comparación con NULL mediante = no evalúa a verdadero.

### Ejercicio 2 — RIGHT JOIN y FULL OUTER JOIN
Ejecute las celdas 4 y 5 para identificar productos sin venta y comparar agosto frente a septiembre.

**Celda 4 de 12**

In [ ]:
# RIGHT JOIN y FULL OUTER JOIN requieren SQLite 3.39.0 o posterior
consulta = """SELECT sqlite_version() AS version_sqlite;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT pr.id_producto, pr.nombre_producto, pr.categoria
FROM detalles_pedido AS d
RIGHT JOIN productos AS pr ON d.id_producto = pr.id_producto
WHERE d.id_detalle IS NULL
ORDER BY pr.id_producto;"""
display(pd.read_sql_query(consulta, conexion))

  version_sqlite
0         3.45.1

   id_producto           nombre_producto   categoria
0            5  Impresora multifuncional     Cómputo
1           14         Reloj inteligente  Accesorios

**Celda 5 de 12**

In [ ]:
conexion.executescript("""
DROP VIEW IF EXISTS resumen_agosto;
DROP VIEW IF EXISTS resumen_septiembre;
CREATE TEMP VIEW resumen_agosto AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-08-01' AND '2026-08-31'
GROUP BY id_cliente;
CREATE TEMP VIEW resumen_septiembre AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-09-01' AND '2026-09-30'
GROUP BY id_cliente;
""")
consulta = """SELECT COALESCE(a.id_cliente, s.id_cliente) AS id_cliente,
       a.monto AS monto_agosto, s.monto AS monto_septiembre
FROM resumen_agosto AS a
FULL OUTER JOIN resumen_septiembre AS s ON a.id_cliente = s.id_cliente
ORDER BY id_cliente;"""
pd.read_sql_query(consulta, conexion)

    id_cliente  monto_agosto  monto_septiembre
0            1        1435.0            1967.0
1            2        1522.0            2615.0
2            3         656.0             840.0
3            4        2452.0               NaN
4            5           NaN             640.0
5            7         165.0            3460.0
6            8         271.0             236.0
7            9         220.0               NaN
8           10        6400.0             723.0
9           12           NaN            1420.0
10          13        5879.0               NaN
11          14         618.0             140.0
12          16        2408.0             110.0
13          17        2339.0             885.0
14          18           NaN            3045.0
15          19        1380.0             122.0
16          21         152.0             160.0
17          23         815.0             200.0

**Pregunta 3.** La copia muestra **SQLite 3.45.1**; esta versión puede cambiar en Colab. La tabla derecha es **productos**. Los artículos sin venta son **Impresora multifuncional (ID 5)** y **Reloj inteligente (ID 14)**. RIGHT JOIN mantiene todos los productos y WHERE d.id_detalle IS NULL filtra los que no aparecen en los detalles.

**Pregunta 4.** Compraron **3 clientes solo en agosto** (ID 4, 9, 13); **3 solo en septiembre** (ID 5, 12, 18), y **12 en ambos meses**. Un NULL en monto_septiembre indica que no hubo pedidos allí, un NULL en monto_agosto que no los hubo en agosto; sin NULL en ambos campos hubo compras en los dos meses. Total: **18 clientes compradores**.

## B. Agrupar y filtrar grupos
### Ejercicio 3 — WHERE, GROUP BY y HAVING
Ejecute la celda 6 para identificar clientes con dos o más pedidos en septiembre.

**Celda 6 de 12**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre,
       COUNT(p.id_pedido) AS pedidos_septiembre,
       SUM(p.total) AS monto_septiembre
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.fecha_pedido >= '2026-09-01'
GROUP BY c.id_cliente, c.nombre
HAVING COUNT(p.id_pedido) >= 2
ORDER BY monto_septiembre DESC, c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

   id_cliente             nombre  pedidos_septiembre  monto_septiembre
0           2     Carlos Mendoza                   3              2615
1           1      Lucía Paredes                   2              1967
2          12      Sofía Linares                   2              1420
3          17        Luis Távara                   2               885
4          10  Patricia Zevallos                   2               723

**Pregunta 5.** WHERE filtra las filas de **septiembre** antes de formar grupos. GROUP BY reúne los pedidos por identificador de cliente y HAVING COUNT(p.id_pedido) >= 2 conserva solo los clientes con **al menos dos pedidos en septiembre**. Tener tres pedidos en total y solo uno en septiembre **no basta**. Aparecen **5 clientes**: Carlos Mendoza (ID 2), Lucía Paredes, Sofía Linares, Luis Távara y Patricia Zevallos.

## C. Consultar y reutilizar resultados
### Ejercicio 4 — Subconsulta: pedidos mayores al promedio
La subconsulta obtiene el promedio por pedido y la consulta exterior encuentra los pedidos cuyo total es superior.

**Celda 7 de 12**

In [ ]:
consulta = """SELECT p.id_pedido, p.fecha_pedido, c.id_cliente, c.nombre, p.total
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente
WHERE p.total > (SELECT AVG(total) FROM pedidos)
ORDER BY p.total DESC, p.id_pedido;"""
pd.read_sql_query(consulta, conexion)

    id_pedido fecha_pedido  id_cliente             nombre  total
0           3   2026-08-02          10  Patricia Zevallos   6400
1          13   2026-08-17          13         Diego Alva   5655
2          26   2026-09-08           7     Carlos Mendoza   3460
3          31   2026-09-15          18          Ana Girón   3045
4          35   2026-09-24           2     Carlos Mendoza   2544
5          20   2026-08-21           4      Jorge Salazar   2452
6           9   2026-08-12          16       Camila Neyra   2256
7          15   2026-08-18          17        Luis Távara   1680
8          11   2026-08-16           2     Carlos Mendoza   1522
9          28   2026-09-10           1      Lucía Paredes   1485
10          7   2026-08-09          19    Pedro Seminario   1380
11          4   2026-08-07           1      Lucía Paredes   1240
12         32   2026-09-17          12      Sofía Linares   1200

**Pregunta 7.** La subconsulta devuelve el promedio de **42 pedidos**, aproximadamente **S/ 1 030,36** (S/ 43 275 / 42). La consulta muestra **13 pedidos** superiores. Lucía Paredes (ID 1) aparece en dos filas y Carlos Mendoza (ID 2) también; otro Carlos Mendoza (ID 7) es **una persona distinta** aunque tenga el mismo nombre.

### Ejercicio 5 — Vista de resumen por cliente
La vista guarda una consulta reutilizable sobre pedidos y clientes.

**Celda 8 de 12**

In [ ]:
conexion.executescript("""
DROP VIEW IF EXISTS vista_resumen_cliente;
CREATE VIEW vista_resumen_cliente AS
SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       COALESCE(SUM(p.total), 0) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad;
""")
consulta = """SELECT *
FROM vista_resumen_cliente
WHERE nombre = 'Carlos Mendoza' OR ciudad = 'Piura'
ORDER BY ciudad, monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

   id_cliente           nombre    ciudad  num_pedidos  monto_acumulado  \
0           7   Carlos Mendoza  Arequipa            2             3625   
1           2   Carlos Mendoza      Lima            4             4137   
2          17      Luis Távara     Piura            4             3224   
3          18        Ana Girón     Piura            1             3045   
4          19  Pedro Seminario     Piura            2             1502   
5          20    Marta Infante     Piura            0                0   
6          24      Natalia Ato     Piura            0                0   

   promedio_pedido  
0          1812.50  
1          1034.25  
2           806.00  
3          3045.00  
4           751.00  
5              NaN  
6              NaN  

**Pregunta 8.** La vista almacena **la definición de la consulta SQL**, no una copia materializada del resultado. El **id_cliente** distingue a Carlos Mendoza de Lima (ID 2, **S/ 4 137**) y Carlos Mendoza de Arequipa (ID 7, **S/ 3 625**). Agrupar por nombre mezclaría indebidamente dos personas.

## Actividad 3 — Informe analítico comparado
Para el top 5 se suma pedidos.total **una sola vez por id_pedido**, directamente desde pedidos. No se suma después de unir a detalles_pedido, porque las varias líneas multiplicarían el total de un mismo pedido.

**Celda 9 de 12**

In [ ]:
# Clientes con mayor monto acumulado (top cinco)
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       SUM(p.total) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY monto_acumulado DESC, c.id_cliente
LIMIT 5;"""
pd.read_sql_query(consulta, conexion)

   id_cliente             nombre    ciudad  num_pedidos  monto_acumulado  \
0          10  Patricia Zevallos  Arequipa            3             7123   
1          13         Diego Alva  Trujillo            2             5879   
2           2     Carlos Mendoza      Lima            4             4137   
3           7     Carlos Mendoza  Arequipa            2             3625   
4           1      Lucía Paredes      Lima            4             3402   

   promedio_pedido  
0          2374.33  
1          2939.50  
2          1034.25  
3          1812.50  
4           850.50  

### Cinco clientes con mayor monto acumulado

| Puesto | ID | Cliente | Ciudad | Pedidos | Acumulado (S/) | Promedio por pedido (S/) |
|---:|---:|---|---|---:|---:|---:|
| 1 | 10 | Patricia Zevallos | Arequipa | 3 | 7 123,00 | 2 374,33 |
| 2 | 13 | Diego Alva | Trujillo | 2 | 5 879,00 | 2 939,50 |
| 3 | 2 | Carlos Mendoza | Lima | 4 | 4 137,00 | 1 034,25 |
| 4 | 7 | Carlos Mendoza | Arequipa | 2 | 3 625,00 | 1 812,50 |
| 5 | 1 | Lucía Paredes | Lima | 4 | 3 402,00 | 850,50 |

**A.** Por acumulado lidera **Patricia Zevallos**, con **S/ 7 123** en tres pedidos. Por **promedio por pedido** destaca **Diego Alva**, con **S/ 2 939,50** (dos pedidos). Los dos Carlos Mendoza se distinguen por ID.

**B.** Antes de recomendar beneficios se necesitan costos, margen, devoluciones, descuentos, recurrencia, periodos de comparación y resultados de promociones. Conviene ensayar un incentivo controlado y medir margen incremental. **Mayor ingreso no demuestra mayor rentabilidad**; estos datos simulados de agosto y septiembre de 2026 no permiten inferir tendencias.

## Retos y retroalimentación
Ejecute las celdas 10 a 12; cada salida se interpreta a continuación.

**Celda 10 de 12**

In [ ]:
consulta = """SELECT c.ciudad, COUNT(c.id_cliente) AS clientes_sin_pedidos
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
GROUP BY c.ciudad
ORDER BY clientes_sin_pedidos DESC, c.ciudad;"""
pd.read_sql_query(consulta, conexion)

     ciudad  clientes_sin_pedidos
0  Arequipa                     2
1     Piura                     2
2      Lima                     1
3  Trujillo                     1

**Reto 1 — interpretación.** Hay **6 clientes sin pedidos**: Arequipa 2, Piura 2, Lima 1 y Trujillo 1. LEFT JOIN mantiene a los clientes sin pareja y WHERE p.id_pedido IS NULL los identifica; GROUP BY cuenta por ciudad. No necesariamente son clientes perdidos.

### Reto 2 — Ingresos por categoría
Se utiliza SUM(cantidad * precio_unitario) sobre el detalle, no SUM(pedidos.total) tras unir detalles.

**Celda 11 de 12**

In [ ]:
consulta = """SELECT pr.categoria,
       SUM(d.cantidad) AS unidades,
       SUM(d.cantidad * d.precio_unitario) AS ingreso
FROM productos AS pr
INNER JOIN detalles_pedido AS d ON pr.id_producto = d.id_producto
GROUP BY pr.categoria
HAVING SUM(d.cantidad * d.precio_unitario) >= 10000
ORDER BY ingreso DESC, pr.categoria;"""
pd.read_sql_query(consulta, conexion)

   categoria  unidades  ingreso
0  Celulares        12    18805
1    Cómputo        37    17662

**Reto 2 — interpretación.** Superan el umbral **Celulares: 12 unidades y S/ 18 805** y **Cómputo: 37 unidades y S/ 17 662**. Las categorías ausentes de la salida no alcanzaron el umbral o no tuvieron ventas. Para identificar **todas** las excluidas por nombre y monto debe consultarse el resumen **sin HAVING**; no se infiere su total a partir de esta salida. HAVING filtra por un valor **agregado del grupo**; WHERE filtra filas individuales.

### Reto 3 — Clientes que superan el promedio acumulado por cliente
La subconsulta considera únicamente clientes con pedidos registrados.

**Celda 12 de 12**

In [ ]:
consulta = """SELECT id_cliente, nombre, ciudad, monto_acumulado
FROM vista_resumen_cliente
WHERE monto_acumulado > (SELECT AVG(monto_acumulado)
                         FROM vista_resumen_cliente
                         WHERE num_pedidos > 0)
ORDER BY monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

   id_cliente             nombre    ciudad  monto_acumulado
0          10  Patricia Zevallos  Arequipa             7123
1          13         Diego Alva  Trujillo             5879
2           2     Carlos Mendoza      Lima             4137
3           7     Carlos Mendoza  Arequipa             3625
4           1      Lucía Paredes      Lima             3402
5          17        Luis Távara     Piura             3224
6          18          Ana Girón     Piura             3045
7          16       Camila Neyra  Trujillo             2518
8           4      Jorge Salazar      Lima             2452

**Reto 3 — interpretación.** La subconsulta devuelve el promedio del acumulado de los **18 clientes que compraron**, aproximadamente **S/ 2 404,17** (S/ 43 275 / 18). **9 clientes** superan esa cifra. No se compara con el promedio por pedido (S/ 1 030,36), porque un indicador usa clientes y el otro pedidos.

## Conclusiones

1. **Integridad en JOIN:** INNER JOIN reporta 42 pedidos de 18 clientes; LEFT JOIN identifica 6 sin pedidos. RIGHT JOIN permite detectar productos no vendidos; FULL OUTER JOIN muestra clientes activos en uno o ambos meses.
2. **Agregaciones bien definidas:** WHERE filtra filas, HAVING filtra agrupaciones. El top cinco suma cada total de pedido una sola vez y distingue clientes por identificador; Patricia lidera por ingresos acumulados, Diego por promedio de pedido.
3. **Limitaciones comerciales:** las cifras provienen de 24 clientes y 42 pedidos **simulados** en agosto y septiembre de 2026. No prueban rentabilidad ni tendencias generales; para tomar decisiones se requieren costos, márgenes y periodos reales más amplios.

**Entrega:** subir este archivo .ipynb a semana03/ del repositorio del equipo. Revisar la accesibilidad y mantener visibles el código, las salidas y las interpretaciones. El cuestionario individual se completa por separado, una vez conocidas sus cuatro preguntas.